In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from scipy.stats import ttest_rel, zscore
from config import *
from load_data import load_both

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

OUT_DIR = REPO_ROOT / 'results' / 'figs' / 'paper'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load both samples
exp_data, conf_data = load_both()
all_data = {'exploratory': exp_data, 'confirmatory': conf_data}
all_data = {k: v for k, v in all_data.items() if v is not None}

for name, d in all_data.items():
    s = d['config']
    print(f'{s.label}: {len(d["beh"])} trials, {d["beh"]["subj"].nunique()} subjects')


## H1a — Choice decreases with threat and distance

**Model:** `choice ~ threat_z + dist_z + threat_z:dist_z` (cluster-robust SE by subject)  
**Test:** β(threat) < 0 AND β(distance) < 0, both p < 0.01

In [ ]:
# ── H1a: Logistic model — choice ~ threat + distance ──
h1a_results = {}

for name, d in all_data.items():
    beh = d['beh_slim']
    label = d['config'].label

    model = smf.logit(
        "choice ~ threat_z + dist_z + threat_z:dist_z",
        data=beh
    ).fit(disp=False, cov_type='cluster', cov_kwds={'groups': beh['subj']})

    b_threat = model.params['threat_z']
    b_dist = model.params['dist_z']
    p_threat = model.pvalues['threat_z']
    p_dist = model.pvalues['dist_z']
    pass_threat = b_threat < 0 and p_threat < 0.01
    pass_dist = b_dist < 0 and p_dist < 0.01

    h1a_results[name] = {
        'label': label,
        'model': model,
        'b_threat': b_threat, 'p_threat': p_threat, 'pass_threat': pass_threat,
        'b_dist': b_dist, 'p_dist': p_dist, 'pass_dist': pass_dist,
        'pass': pass_threat and pass_dist,
    }

    print(f"H1a — {label}")
    print("=" * 65)
    print(model.summary2().tables[1][['Coef.', 'Std.Err.', 'z', 'P>|z|']].to_string())
    print()
    print(f"  Threat:   β = {b_threat:.4f}, p = {p_threat:.2e}  {'PASS' if pass_threat else 'FAIL'}")
    print(f"  Distance: β = {b_dist:.4f}, p = {p_dist:.2e}  {'PASS' if pass_dist else 'FAIL'}")
    print()


In [ ]:
# ── H1a figure: 3×3 Choice Surface (side by side) ──
n_samples = len(all_data)
fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 4))
if n_samples == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, all_data.items()):
    beh = d['beh_slim']
    label = d['config'].label

    choice_surface = beh.groupby(['T_round', 'distance_H'])['choice'].mean().unstack()

    im = ax.imshow(choice_surface.values, cmap='RdYlGn', vmin=0, vmax=1,
                   origin='lower', aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(['D=1', 'D=2', 'D=3'])
    ax.set_yticks(range(3)); ax.set_yticklabels(['T=0.1', 'T=0.5', 'T=0.9'])
    ax.set_xlabel('Escape Distance'); ax.set_ylabel('Threat Probability')
    ax.set_title(f'P(Choose High-Effort)\n{label}')
    for i in range(3):
        for j in range(3):
            val = choice_surface.values[i, j]
            color = 'white' if val < 0.4 or val > 0.8 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color=color, fontweight='bold')
    plt.colorbar(im, ax=ax, label='P(Heavy)')

plt.tight_layout()
plt.savefig(OUT_DIR / "H1a_choice_surface.png", dpi=150, bbox_inches='tight')
plt.show()


## H1b — Anxiety increases and confidence decreases with threat

**Model:** `response ~ threat_z + (1 + threat_z | subj)` separately for anxiety and confidence  
**Test:** β > 0 for anxiety, β < 0 for confidence, both |t| > 3.0

In [ ]:
# ── H1b: Affect ~ threat ──
h1b_results = {}

for name, d in all_data.items():
    feelings = d['feelings']
    label = d['config'].label

    anx = feelings[feelings['questionLabel'] == 'anxiety'].copy()
    con = feelings[feelings['questionLabel'] == 'confidence'].copy()
    anx['T_z'] = zscore(anx['threat'].astype(float))
    con['T_z'] = zscore(con['threat'].astype(float))

    print(f"H1b — {label}")
    print("=" * 55)

    sample_results = {}
    for affect_label, df in [('Anxiety', anx), ('Confidence', con)]:
        try:
            m = smf.mixedlm(
                "response ~ T_z", data=df, groups=df['subj'],
                re_formula="~T_z"
            ).fit(reml=False)
        except Exception:
            m = smf.mixedlm(
                "response ~ T_z", data=df, groups=df['subj']
            ).fit(reml=False)

        b = m.fe_params['T_z']
        z = m.tvalues['T_z']
        p = m.pvalues['T_z']
        if affect_label == 'Anxiety':
            passed = b > 0 and abs(z) > 3.0
        else:
            passed = b < 0 and abs(z) > 3.0

        sample_results[affect_label] = {'b': b, 'z': z, 'p': p, 'pass': passed}

        print(f"  {affect_label:12s}: β = {b:.4f}, z = {z:.3f}, p = {p:.2e}")
        print(f"               {'PASS' if passed else 'FAIL'} (expect {'positive' if affect_label == 'Anxiety' else 'negative'} β, |t| > 3)")
        print()

    h1b_results[name] = {
        'label': label,
        'pass': sample_results['Anxiety']['pass'] and sample_results['Confidence']['pass'],
        **sample_results,
    }


## H1c — Vigor increases with threat

Normalized press rate (median(1/IPI) / calibrationMax) increases with threat probability. The effect is small in absolute scale but robust.

**Test:** LMM: `median_rate ~ threat_z + is_heavy + (1 | subj)`, controlling for cookie type. β(threat) > 0, p < 0.01.

In [ ]:
# ── H1c: Vigor ~ threat (LMM, controlling for cookie type) ──
h1c_results = {}

for name, d in all_data.items():
    vigor_valid = d['vigor_valid']
    label = d['config'].label

    print(f"H1c — {label}")
    print("=" * 55)

    # Mean vigor by threat level
    print("Normalized press rate by threat level:")
    means = vigor_valid.groupby('T_round')['norm_rate'].agg(['mean', 'sem'])
    for t, row in means.iterrows():
        print(f"  T={t}: {row['mean']:.3f} (SE={row['sem']:.4f})")
    print()

    # LMM controlling for cookie type
    model = smf.mixedlm(
        "norm_rate ~ threat_z + is_heavy",
        data=vigor_valid,
        groups=vigor_valid['subj']
    ).fit(reml=False)

    b = model.fe_params['threat_z']
    z = model.tvalues['threat_z']
    p = model.pvalues['threat_z']
    passed = b > 0 and p < 0.01

    h1c_results[name] = {
        'label': label,
        'b': b, 'z': z, 'p': p,
        'b_heavy': model.fe_params['is_heavy'],
        'pass': passed,
    }

    print(f"  threat_z: β = {b:.4f}, z = {z:.3f}, p = {p:.2e}")
    print(f"  is_heavy: β = {model.fe_params['is_heavy']:.4f}")
    print(f"  Verdict: {'PASS' if passed else 'FAIL'} (expect positive β, p < 0.01)")
    print()


In [ ]:
# ── H1c figure: Vigor by threat level (side by side) ──
n_samples = len(all_data)
fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 4))
if n_samples == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, all_data.items()):
    vigor_valid = d['vigor_valid']
    label = d['config'].label

    means = vigor_valid.groupby('T_round')['norm_rate'].agg(['mean', 'sem'])
    ax.bar(means.index, means['mean'], width=0.25,
           color=['forestgreen', 'gold', 'firebrick'], edgecolor='white',
           yerr=means['sem'] * 1.96, capsize=3)
    ax.set_xlabel('Threat Probability')
    ax.set_ylabel('Normalized Press Rate')
    ax.set_title(f'H1c: Vigor by Threat\n{label}')
    ax.set_xticks([0.1, 0.5, 0.9])
    for t, row in means.iterrows():
        ax.text(t, row['mean'] + row['sem'] * 2.2, f"{row['mean']:.3f}",
                ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "H1c_vigor_by_threat.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── H1 Summary Table ──
print("=" * 70)
print("H1 SUMMARY — Pass/Fail by Sample")
print("=" * 70)

rows = []
for name in all_data:
    label = all_data[name]['config'].label
    rows.append({
        'Sample': label,
        'H1a (choice)': 'PASS' if h1a_results[name]['pass'] else 'FAIL',
        'H1b (affect)': 'PASS' if h1b_results[name]['pass'] else 'FAIL',
        'H1c (vigor)':  'PASS' if h1c_results[name]['pass'] else 'FAIL',
    })

summary = pd.DataFrame(rows).set_index('Sample')
print(summary.to_string())
print()

# Overall
for name in all_data:
    label = all_data[name]['config'].label
    all_pass = (h1a_results[name]['pass']
                and h1b_results[name]['pass']
                and h1c_results[name]['pass'])
    status = 'ALL PASS' if all_pass else 'SOME FAIL'
    print(f"  {label}: {status}")
